# Session 1: position, velocity and acceleration are one story
**A. Real-world problem.** A car pulls away from a junction, cruises, then brakes. Your phone
or a speed camera records where it is at each moment. From that one record we want to know how
fast it was going and whether it was speeding up or slowing down.

**Plan.** (1) write the three quantities for constant acceleration, (2) measure the slope of
the position graph and compare it with velocity, (3) measure the area under the velocity graph
and compare it with the distance moved, (4) draw the three linked graphs.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the three quantities
* position x (metres): where the car is on the road,
* velocity v (metres per second): how fast x changes, and in which direction,
* acceleration a (metres per second squared): how fast v changes.

For a constant acceleration a, starting at x0 with velocity v0 at time t = 0:

    a(t) = a        v(t) = v0 + a t        x(t) = x0 + v0 t + (1/2) a t^2

Check by hand with x0 = 0, v0 = 4 m/s, a = 2 m/s^2 at t = 5 s:
v = 4 + 2*5 = 14 m/s and x = 4*5 + 0.5*2*25 = 45 m.

In [ ]:
def position(t, x0=0.0, v0=0.0, a=0.0):
    t = np.asarray(t, dtype=float)
    return x0 + v0 * t + 0.5 * a * t ** 2


def velocity(t, v0=0.0, a=0.0):
    t = np.asarray(t, dtype=float)
    return v0 + a * t


def acceleration(t, a=0.0):
    t = np.asarray(t, dtype=float)
    return a + 0.0 * t

## Step 2: slope and area
The slope of a graph is "rise over run". For the x-t graph the rise is a change in position
and the run is a change in time, so the slope is a velocity. Likewise the slope of the v-t
graph is an acceleration. Going the other way, the area under the v-t graph is
(velocity) times (time) = a distance, and the area under the a-t graph is a change of velocity.

In [ ]:
def slope(f, t, h=1e-3):
    """Slope of the graph of f at time t, from two points a tiny step h either side."""
    return float((f(t + h) - f(t - h)) / (2 * h))


def area(f, t0, t1, n=2000):
    """Area under the graph of f between t0 and t1 (trapezoid rule). Area below the axis counts negative."""
    t = np.linspace(t0, t1, n + 1)
    y = np.asarray(f(t), dtype=float)
    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(t)))


def stop_time(v0, a):
    """Time when the velocity becomes zero (needs a != 0 and v0, a of opposite sign)."""
    return -v0 / a

## Step 3: see it
Three graphs stacked with the same time axis. Where the x-t curve is steep the v-t graph is
high; where v-t is above zero, x-t rises. The shaded area under v-t up to 5 s is the 45 m.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    x0, v0, a = 0.0, 4.0, 2.0
    t = np.linspace(0, 8, 321)
    ts = 5.0
    xs, vs = float(position(ts, x0, v0, a)), float(velocity(ts, v0, a))
    print(f"t = {ts} s: x = {xs:.1f} m, v = {vs:.1f} m/s")
    print("slope of x-t at 5 s:", round(slope(lambda u: position(u, x0, v0, a), ts), 3), "m/s")
    print("area under v-t, 0 to 5 s:", round(area(lambda u: velocity(u, v0, a), 0, ts), 3), "m")

    fig, ax = plt.subplots(3, 1, figsize=(6, 7), sharex=True)
    ax[0].plot(t, position(t, x0, v0, a), color="C0")
    tang = xs + vs * (t - ts)
    ax[0].plot(t, tang, "k--", lw=1, label=f"tangent: slope {vs:.0f} m/s")
    ax[0].plot([ts], [xs], "ko")
    ax[0].set_ylim(-5, 110)
    ax[0].set_ylabel("position x (m)")
    ax[0].legend(loc="upper left")
    ax[1].plot(t, velocity(t, v0, a), color="C1")
    tt = np.linspace(0, ts, 100)
    ax[1].fill_between(tt, velocity(tt, v0, a), alpha=0.25, color="C1", label="area = 45 m")
    ax[1].set_ylabel("velocity v (m/s)")
    ax[1].legend(loc="upper left")
    ax[2].plot(t, acceleration(t, a), color="C2")
    ax[2].set_ylim(0, 4)
    ax[2].set_ylabel("acceleration a (m/s^2)")
    ax[2].set_xlabel("time t (s)")
    for axis in ax:
        axis.grid(alpha=0.3)
    ax[0].set_title("Car with v0 = 4 m/s and a = 2 m/s^2")
    fig.tight_layout()
    fig.savefig(OUT / "three_graphs.png", dpi=120)

    # braking: the car stops, and the equations then say it reverses
    v0b, ab = 10.0, -2.0
    tb = np.linspace(0, 8, 321)
    tstop = stop_time(v0b, ab)
    print(f"braking from {v0b} m/s at {ab} m/s^2: stops at t = {tstop} s, x = {float(position(tstop, 0, v0b, ab)):.1f} m")
    fig2, ax2 = plt.subplots(1, 2, figsize=(8, 3.5))
    ax2[0].plot(tb, position(tb, 0, v0b, ab))
    ax2[0].axvline(tstop, color="gray", ls=":")
    ax2[0].set_xlabel("time t (s)")
    ax2[0].set_ylabel("position x (m)")
    ax2[0].set_title("x-t: turns round at the stop")
    ax2[1].plot(tb, velocity(tb, v0b, ab), color="C1")
    ax2[1].axhline(0, color="k", lw=0.8)
    ax2[1].axvline(tstop, color="gray", ls=":")
    ax2[1].set_xlabel("time t (s)")
    ax2[1].set_ylabel("velocity v (m/s)")
    ax2[1].set_title("v-t: crosses zero at 5 s")
    for axis in ax2:
        axis.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "braking.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change `v0` and `a` in `lesson.py` to 6 m/s and -1.5 m/s^2. Predict the velocity and position at 4 s by hand before you run it.
2. **Stretch.** Write `average_velocity(x0, x1, t0, t1)` that returns the slope between two points of an x-t graph, and use it on the car of the lesson between 2 s and 6 s. Compare with the velocity at 4 s.
3. **Explain.** Explain, in your own words, why the area under a velocity-time graph is a distance and why part of the area can count as negative.

## Assessment
- **A. Conceptual.** What does the slope of a position-time graph represent? What does a negative slope mean, and what does a flat part mean?
- **B. Numerical.** A car has v0 = 3 m/s and a = 2.5 m/s^2. Find its velocity and the distance it has moved after 4 s. Check the units and that the answer is sensible.
- **C. Graph and simulation interpretation.** A v-t graph is a straight line from (0 s, 12 m/s) down to (6 s, 0 m/s). In which direction is the acceleration, what is its value, and how far did the body move?
- **D. Design and optimisation challenge.** Design: a car travelling at 10 m/s must stop within 30 m. What constant deceleration is needed at least, and how long does the stop take? State your assumptions.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    x5, v5 = float(L.position(5, 0, 4, 2)), float(L.velocity(5, 4, 2))
    checks.append(("worked example: v0 = 4 m/s, a = 2 m/s^2 gives x = 45 m and v = 14 m/s at 5 s",
                   abs(x5 - 45.0) < 1e-9 and abs(v5 - 14.0) < 1e-9))
    checks.append(("slope of x-t at 5 s equals the velocity 14 m/s",
                   abs(L.slope(lambda u: L.position(u, 0, 4, 2), 5.0) - 14.0) < 1e-6))
    checks.append(("area under v-t from 0 to 5 s equals the distance 45 m",
                   abs(L.area(lambda u: L.velocity(u, 4, 2), 0, 5) - 45.0) < 1e-6))
    checks.append(("slope of v-t equals the acceleration 2 m/s^2",
                   abs(L.slope(lambda u: L.velocity(u, 4, 2), 3.0) - 2.0) < 1e-6))
    ts = L.stop_time(10, -2)
    checks.append(("hand calculation: braking from 10 m/s at -2 m/s^2 stops at 5 s after 25 m",
                   abs(ts - 5.0) < 1e-12 and abs(float(L.position(ts, 0, 10, -2)) - 25.0) < 1e-9))
    checks.append(("edge: a = 0 gives constant velocity and a straight x-t line",
                   np.allclose(L.velocity([0, 3, 9], 5, 0), 5) and abs(float(L.position(3, 1, 5, 0)) - 16.0) < 1e-12))
    checks.append(("area below the axis counts negative: v = -3 m/s for 4 s is -12 m",
                   abs(L.area(lambda u: L.velocity(u, -3, 0), 0, 4) + 12.0) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
